<a href="https://colab.research.google.com/github/nehvarsu/Fraud_Detection/blob/main/Agentic-Fraud-Detection-XGBoost-LLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
from google.colab import userdata

# 1. Load credentials securely
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY') # CrewAI will use this

# 2. Install required modern AI and ML libraries
!pip install -q kaggle google-generativeai crewai xgboost scikit-learn

# 3. Download and extract the PaySim1 dataset
!kaggle datasets download -d rizalkurnia198/ieee-cis-fraud-detection-memory-optimized-dataset
!kaggle datasets download -d rizalkurnia198/ieee-cis-fraud-detection-memory-optimized-dataset
!unzip -q ieee-cis-fraud-detection-memory-optimized-dataset.zip -d ieee_data
print("Environment ready and dataset downloaded.")

Dataset URL: https://www.kaggle.com/datasets/rizalkurnia198/ieee-cis-fraud-detection-memory-optimized-dataset
License(s): unknown
ieee-cis-fraud-detection-memory-optimized-dataset.zip: Skipping, found more recently modified local copy (use --force to force download)
Dataset URL: https://www.kaggle.com/datasets/rizalkurnia198/ieee-cis-fraud-detection-memory-optimized-dataset
License(s): unknown
ieee-cis-fraud-detection-memory-optimized-dataset.zip: Skipping, found more recently modified local copy (use --force to force download)
replace ieee_data/data_optimized.parquet? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
Environment ready and dataset downloaded.


In [2]:
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

# 1. Load the specific parquet file found in the directory
file_path = 'ieee_data/data_optimized.parquet'
print(f"Loading data from: {file_path}")
df = pd.read_parquet(file_path)

# 2. Take a 2,000-row sample for rapid prototyping (THIS CREATES df_sample)
df_sample = df.sample(n=2000, random_state=42)

# 3. Dynamically map column names
expected_features = ['TransactionAmt', 'card1', 'card2', 'card3', 'card5', 'addr1', 'addr2']
available_features = [col for col in expected_features if col in df_sample.columns]

if not available_features:
    lower_cols = [c.lower() for c in df_sample.columns]
    df_sample.columns = lower_cols
    available_features = [col.lower() for col in expected_features if col.lower() in df_sample.columns]

print(f"Using features: {available_features}")
X = df_sample[available_features].fillna(-999)

target_col = 'isFraud' if 'isFraud' in df_sample.columns else 'isfraud'
y = df_sample[target_col]

# 4. Train the baseline model
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y)

xgb_model = xgb.XGBClassifier(eval_metric='auc', use_label_encoder=False)
xgb_model.fit(X_train, y_train)

auc = roc_auc_score(y_test, xgb_model.predict_proba(X_test)[:, 1])
print(f"Baseline XGBoost Trained. AUC Score: {auc:.4f}")

Loading data from: ieee_data/data_optimized.parquet
Using features: ['TransactionAmt', 'card1', 'card2', 'card3', 'card5', 'addr1', 'addr2']


/usr/local/lib/python3.13/dist-packages/xgboost/training.py:200: UserWarning: [00:57:19] WARNING: /__w/xgboost/xgboost/src/learner.cc:794: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Baseline XGBoost Trained. AUC Score: 0.6701


Phase 1: Free-Tier Vector Embeddings

In [3]:
import time
import requests
from google.colab import userdata
from langchain_google_genai import GoogleGenerativeAIEmbeddings

# 1. Fetch key directly to avoid runtime disconnect errors
gemini_key = userdata.get('GEMINI_API_KEY')

# --- 2. DYNAMICALLY DISCOVER MODEL VIA RAW API REQUEST ---
# This bypasses all Google SDK version conflicts in Colab
print("Querying API for authorized embedding models...")
url = f"https://generativelanguage.googleapis.com/v1beta/models?key={gemini_key}"
response = requests.get(url).json()

embedding_model_name = None
for model in response.get('models', []):
    if 'embedContent' in model.get('supportedGenerationMethods', []):
        embedding_model_name = model['name']
        # Stop at the first valid embedding model found
        break

if not embedding_model_name:
    raise ValueError("No embedding models found for this API key.")

print(f"Dynamically discovered embedding model: {embedding_model_name}")

# --- 3. INITIALIZE LANGCHAIN EMBEDDINGS ---
embeddings_model = GoogleGenerativeAIEmbeddings(
    model=embedding_model_name,
    google_api_key=gemini_key
)

# --- 4. PREPARE THE DATA ---
col_amt = 'TransactionAmt' if 'TransactionAmt' in df_sample.columns else 'transactionamt'
col_card = 'card4'
col_email = 'P_emaildomain' if 'P_emaildomain' in df_sample.columns else 'p_emaildomain'

df_sample[col_card] = df_sample[col_card].astype(object).fillna("unknown")
df_sample[col_email] = df_sample[col_email].astype(object).fillna("unknown")

df_sample['context_memo'] = (
    "Transaction of $" + df_sample[col_amt].astype(str) +
    " via " + df_sample[col_card].astype(str) +
    " card, email domain: " + df_sample[col_email].astype(str)
)

# --- 5. GENERATE EMBEDDINGS (WITH GUARDRAILS) ---
MAX_ROWS = 15
df_subset = df_sample.head(MAX_ROWS).copy()
embeddings_list = []

print("Generating embeddings. The 4-second pause ensures you are never rate-limited...")
for index, row in df_subset.iterrows():
    try:
        time.sleep(4)
        vector = embeddings_model.embed_query(row['context_memo'])
        embeddings_list.append(vector)
        print(f"✅ Row vectorized: {row['context_memo'][:50]}...")
    except Exception as e:
        print(f"⚠️ API Limit/Error: {e}")
        embeddings_list.append(None)

df_subset['text_embeddings'] = embeddings_list
print(f"\nSuccessfully embedded {len([e for e in embeddings_list if e is not None])} rows.")

Querying API for authorized embedding models...
Dynamically discovered embedding model: models/gemini-embedding-001
Generating embeddings. The 4-second pause ensures you are never rate-limited...
✅ Row vectorized: Transaction of $724.0 via mastercard card, email d...
✅ Row vectorized: Transaction of $108.5 via visa card, email domain:...
✅ Row vectorized: Transaction of $47.95 via mastercard card, email d...
✅ Row vectorized: Transaction of $100.599 via visa card, email domai...
✅ Row vectorized: Transaction of $107.95 via visa card, email domain...
✅ Row vectorized: Transaction of $280.0 via mastercard card, email d...
✅ Row vectorized: Transaction of $311.95 via visa card, email domain...
✅ Row vectorized: Transaction of $330.99 via visa card, email domain...
✅ Row vectorized: Transaction of $10.392 via visa card, email domain...
✅ Row vectorized: Transaction of $335.0 via visa card, email domain:...
✅ Row vectorized: Transaction of $57.0 via visa card, email domain: ...
✅ Row vector

Cell 4: Agentic Triage & Explainability
Finally, run the agent to explain the fraud.

In [6]:
# --- 0. ENSURE DEPENDENCIES ARE INSTALLED ---
!pip install -q crewai

import os
from google.colab import userdata
from crewai import Agent, Task, Crew, Process

# 1. Set the API key in the environment
os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')

# --- 2. DEFINE THE AI INVESTIGATOR ---
# We use the exact model string the API error told us to use
compliance_agent = Agent(
    role='Senior AML Compliance Analyst',
    goal='Analyze flagged transactions and determine the specific fraud topology.',
    backstory='You are an expert in financial crime. You explain why a transaction flagged by XGBoost is suspicious based on the transaction amount, card type, and email domain.',
    llm='gemini/gemini-3.8-flash',
    max_iter=3,
    verbose=True
)

# --- 3. PREPARE THE FLAGGED TRANSACTION ---
col_amt = 'TransactionAmt' if 'TransactionAmt' in df_sample.columns else 'transactionamt'
col_card = 'card4'
col_email = 'P_emaildomain' if 'P_emaildomain' in df_sample.columns else 'p_emaildomain'
target_col = 'isFraud' if 'isFraud' in df_sample.columns else 'isfraud'

fraud_rows = df_sample[df_sample[target_col] == 1]

# Failsafe: if the sample has no fraud, use the first row
if len(fraud_rows) > 0:
    flagged_row = fraud_rows.iloc[0]
else:
    flagged_row = df_sample.iloc[0]

flagged_txn_details = flagged_row[[col_amt, col_card, col_email]].to_dict()

# --- 4. DEFINE THE TRIAGE TASK ---
triage_task = Task(
    description=f"XGBoost flagged this transaction: {flagged_txn_details}. Write a 3-sentence compliance narrative explaining why these specific attributes indicate potential fraud.",
    expected_output="A brief, human-readable compliance report for the investigation queue.",
    agent=compliance_agent
)

# --- 5. EXECUTE THE AGENT ---
crew = Crew(
    agents=[compliance_agent],
    tasks=[triage_task],
    process=Process.sequential
)

print("\n--- INITIATING AGENTIC REVIEW ---")
# Tell Colab to safely await the async kickoff
result = await crew.kickoff_async()

print("\n--- FINAL COMPLIANCE REPORT ---")
print(result)


--- INITIATING AGENTIC REVIEW ---


╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior AML Compliance Analyst                                                                           │
│                                                                                                                 │
│  Task: XGBoost flagged this transaction: {'TransactionAmt': 10.392000198364258, 'card4': 'visa',                │
│  'P_emaildomain': 'icloud.com'}. Write a 3-sentence compliance narrative explaining why these specific          │
│  attributes indicate potential fraud.                                                                           │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior AML Compliance Analyst                                                                           │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  The low-value transaction amount of approximately $10.39 on a Visa card strongly suggests an automated         │
│  card-testing topology, commonly utilized by illicit actors to validate stolen payment credentials before       │
│  executing larger fraudulent operations. This risk is further compounded by the association with an             │
│  'icloud.com' email address, a provider frequently exploited through anonymized relay features to conceal       │
│  actor identity and evade velocity-matching controls. Additionally, the irregular fractional floating-point     │
│  amount indicates automated cross-border currency conversion or script-driven checkout behavior, warranting an  │
│  immediate transaction hold and escalation for secondary credential verification.                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


--- FINAL COMPLIANCE REPORT ---
The low-value transaction amount of approximately $10.39 on a Visa card strongly suggests an automated card-testing topology, commonly utilized by illicit actors to validate stolen payment credentials before executing larger fraudulent operations. This risk is further compounded by the association with an 'icloud.com' email address, a provider frequently exploited through anonymized relay features to conceal actor identity and evade velocity-matching controls. Additionally, the irregular fractional floating-point amount indicates automated cross-border currency conversion or script-driven checkout behavior, warranting an immediate transaction hold and escalation for secondary credential verification.
